In [1]:
#|include: false 
#| code-fold: true
#| output: false
#| code-summary: "Library Installation"

%pip install --upgrade openai
%pip install claudette
%pip install python-dotenv
%pip install -U bitsandbytes
%pip install optimum
%pip install auto-gptq
%pip install Wikipedia-API
%pip install tiktoken
%pip install sentence-transformers
%pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.8/644.8 kB 31.8 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.72.0
    Uninstalling openai-1.72.0:
      Successfully uninstalled openai-1.72.0
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
#|include: false 
#| code-fold: true
#| output: false
#| code-summary: "Library Import"

import tokenize, ast
from io import BytesIO
import os

from transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig
import torch
import torch

import ipywidgets as widgets

from openai import OpenAI
from sentence_transformers import SentenceTransformer

In [2]:
d1 = "I’ve been having pain and swelling in my knees and wrists, especially in the morning. It gets a little better as the day goes on. I'm also feeling fatigued and have noticed a rash on my face."

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import PyPDF2
import numpy as np
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F
import re

def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        for page_num in range(len(reader.pages)):
            text += reader.pages[page_num].extract_text() + "\n\n"
    return text

def split_into_chunks(text, chunk_size=500, overlap=50):
    # Clean text first - remove excessive punctuation and page numbers
    text = re.sub(r'\.{3,}', ' ', text)  # Replace sequences of dots with space
    text = re.sub(r'\d{3,4}\s+Uganda\s+Clinical\s+Guidelines\s+\d{4}', '', text)  # Remove page markers
    
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

def setup_rag_system(pdf_path="/teamspace/studios/this_studio/symptom_to_disease/ug_cg_23.pdf"):
    # Load PDF
    guidelines_text = extract_text_from_pdf(pdf_path)
    chunks = split_into_chunks(guidelines_text)
    
    # Create embeddings
    emb_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda" if torch.cuda.is_available() else "cpu")
    
    # Cache embeddings
    chunk_embeddings = []
    for chunk in chunks:
        embedding = emb_model.encode(chunk, convert_to_tensor=True)
        chunk_embeddings.append(embedding)
    
    model = AutoModelForCausalLM.from_pretrained(
        "TheBloke/Llama-2-7b-Chat-GPTQ", 
        device_map=0, 
        torch_dtype=torch.float16
    )
    tokenizer = AutoTokenizer.from_pretrained("TheBloke/Llama-2-7b-Chat-GPTQ")
    
    return {
        "chunks": chunks,
        "chunk_embeddings": chunk_embeddings,
        "model": model,
        "tokenizer": tokenizer,
        "emb_model": emb_model
    }

def retrieve_context(query, rag_system, top_k=5):  # Increased from 3 to 5
    emb_model = rag_system["emb_model"]
    chunks = rag_system["chunks"]
    chunk_embeddings = rag_system["chunk_embeddings"]
    
    # Add medical keywords to enhance the query
    symptoms = query.lower()
    enhanced_query = query
    if "joint" in symptoms or "knee" in symptoms or "wrist" in symptoms:
        enhanced_query += " arthritis rheumatoid joint inflammation"
    elif "ear" in symptoms:
        enhanced_query += " otitis media ear infection"
    elif "skin" in symptoms or "rash" in symptoms:
        enhanced_query += " dermatitis skin condition eczema"
    
    # Encode query
    query_embedding = emb_model.encode(enhanced_query, convert_to_tensor=True)
    
    # Calculate similarities
    similarities = []
    for chunk_emb in chunk_embeddings:
        similarity = F.cosine_similarity(query_embedding, chunk_emb, dim=0)
        similarities.append(similarity.item())
    
    # Get top k chunks
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    return [(chunks[i], similarities[i]) for i in top_indices]

def create_llama_prompt(context, symptoms):
    return f"""<s>[INST] <<SYS>>
You are a medical diagnostic assistant following Ugandan healthcare guidelines.
Your task is to provide a differential diagnosis based on patient symptoms.
Be specific, concise, and thorough in your analysis.
If the symptoms don't match perfectly with the guidelines, still provide your best medical assessment.
<</SYS>>

Based on the following excerpts from the Ugandan Clinical Guidelines and your medical knowledge:

{context}

Provide a differential diagnosis for a patient presenting with these symptoms: {symptoms}

List the top 3-5 possible diagnoses in order of likelihood, with a brief explanation for each.
Even if the guidelines don't directly address these symptoms, use your medical knowledge to make appropriate inferences.
[/INST]"""

def gen(prompt, model, tokenizer, max_length=500):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_length,
        do_sample=True,
        temperature=0.3,  # Lowered further for more focused responses
        top_p=0.85,
        repetition_penalty=1.3,  # Increased to further avoid repetition
        length_penalty=1.0,  # Encourages complete sentences
        no_repeat_ngram_size=3  # Prevent repeating 3-grams
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def diagnose_symptoms(symptoms_text, rag_system, top_k=5):
    # Retrieve relevant context
    relevant_chunks = retrieve_context(symptoms_text, rag_system, top_k)
    
    # Format context - improved to be more structured and cleaner
    context_texts = []
    for i, (chunk, score) in enumerate(relevant_chunks, 1):
        # Remove page numbers and excessive punctuation
        clean_chunk = re.sub(r'\.{3,}', ' ', chunk)  # Replace sequences of dots with space
        clean_chunk = re.sub(r'\d{3,4}\s+Uganda\s+Clinical\s+Guidelines\s+\d{4}', '', clean_chunk)
        clean_chunk = re.sub(r'\s{2,}', ' ', clean_chunk)  # Replace multiple spaces with single space
        
        # Extract any disease names or chapter titles for better context
        disease_match = re.search(r'(CHAPTER \d+[^\.]+|[A-Z][a-z]+\s+[A-Z][a-z]+\s+(Disease|Syndrome|Disorder|Infection|Pain))', clean_chunk)
        section_title = f" - {disease_match.group(0)}" if disease_match else ""
        
        context_texts.append(f"SECTION {i}{section_title}:\n{clean_chunk[:500]}")  # Limit length to keep focused
    
    context = "\n\n".join(context_texts)
    
    # Create prompt with the improved format
    prompt = create_llama_prompt(context, symptoms_text)
    
    # Generate response
    model = rag_system["model"]
    tokenizer = rag_system["tokenizer"]
    full_response = gen(prompt, model, tokenizer, max_length=800)  # Increased length for more complete responses
    
    # Extract just the model's response
    response_parts = full_response.split("[/INST]")
    if len(response_parts) > 1:
        response = response_parts[1].strip()
    else:
        response = full_response
    
    result = {
        "diagnosis": response,
        "context_used": [chunk for chunk, _ in relevant_chunks],
        "similarity_scores": [sim for chunk, sim in relevant_chunks]
    }
    
    # Print results 
    print("\n=== Diagnosis ===")
    print(result["diagnosis"])
    
    print("\n=== Top Relevant Guidelines Used ===")
    for i, (context, score) in enumerate(zip(result["context_used"], result["similarity_scores"]), 1):
        # Print cleaner context summaries
        clean_summary = re.sub(r'\.{3,}', ' ', context[:200])
        print(f"--- Context {i} (Similarity: {score:.3f}) ---")
        print(clean_summary + "..." if len(context) > 200 else clean_summary)
        print()
        
    return result

# Usage remains the same
rag_system = setup_rag_system()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


In [6]:
# Then use diagnose_symptoms with your prompt
result = diagnose_symptoms("Ive been having pain and swelling in my knees and wrists, especially in the morning. It gets a little better as the day goes on. I'm also feeling fatigued and have noticed a rash on my face.", rag_system)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/transformers/generation/utils.py:1532: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(



=== Diagnosis ===
Based on the provided information, here are five potential diagnoses along with their probabilities;
Reactive Arthritisa chronic condition characterized by inflammation in one or more joints triggered by environmental factors such as viruses, bacterial infection, or autoimmune conditions. This could explain why you experience persistent pain and morning stiffness associated with redness and warmth around the affected area. As a result, reactive artritishas a high probability of being responsible for your current symptoms(Uganda National Health Service).
OsteoArthritiso degenerative form of arthrithat occurs when cartilage wears away over time due to aging or repeated stress injuries. Although it typically affects older people, younger individuals who engage in heavy physical activity might develop this condition too. Your knees would likely suffer most because they bear significant bodyweight during daily activities.
Gouta type of arhtritis caused by excess uric acid

In [7]:
# Then use diagnose_symptoms with your prompt
result = diagnose_symptoms("Ear pain, fever, and difficulty hearing", rag_system)


=== Diagnosis ===
Based on the provided information, here are my top 5 differential diagnoses for this patient:
1. Acute Otitis Media (AOM): The presence of ear pain, temperature, and hearing loss suggest an inflamed middle ear caused by a viral or bacteria infection. This condition is common among young children and can be managed with antibacterial medication and observation.
2. Middle Ear Infections (MER): Similar to AOM, MER occurs when the middle ears become infected due to fluid buildup behind the eardrum. It often affects older children and adults more than younger ones. Treatment involves antibiótico therapy and watchful waiting.
3. Ototoxicity: Exposure to loud noises or certain drugs like gentamycin can damage the inner ear structures leading to permanent hearing loss, balance problems, and vertigo. Other signs include nausea, vomiting, and tinnitus. Diagnosis requires audiometry testing and other evaluations.
4. Meniere’s disease: An inner ear disorder characterized by epis

In [8]:
# Then use diagnose_symptoms with your prompt
result = diagnose_symptoms("pain and swelling in my knees and wrists, especially in the morning. It gets a little better as the day goes on. I'm also feeling fatigued and have noticed a rash on my face.", rag_system)


=== Diagnosis ===
Based on the provided information, here are five potential diagnoses along with their probabilities:
Top 3 Most Likely Diagnoses:
Reactive Arthritisa - This condition occurs when the body reacts to an underlying infection or inflammation in the joints, leading to painful swellingsymmetrical distribution around the jointstenderness and warmth in the affected area. As you mentioned, it often presents itself during the morning hours and improves throughout the day. Given this pattern, reactive artritishas a higher probability than any other conditions listed below.
OsteoArthritiso - Similar toreative arthrithisis caused by wear and tear on the joint cartilage over time rather than an infection. Age, genetic predisposition, previous injuries, obesity, etc., could contribute to its development. Kneewill likely experience more severe symptoms compared to thewrist due to greater exposure to stressors such as walking, running, standing, lifting heavy objects, etc.
Gout Attac